# Faster Whisper 語音辨識 → 字幕檔（Colab 版）

依序執行下面三個區塊：

1. **安裝套件**：每次開新的執行階段跑一次。請先到「執行階段 → 變更執行階段類型」選 **T4 GPU**。
2. **掛載雲端硬碟**（選用）：檔案放在 Google Drive、或想把結果存回 Drive 時才需要。
3. **語音辨識**：設定參數後執行。
4. **字幕翻譯**（選用）：上傳中文 `.srt`，翻成英文或中英雙語字幕。可以單獨使用（只要先跑過區塊 1）。

「來源」欄位可以填：
* 留空 → 跳出上傳視窗，從電腦選檔（可多選）
* Colab 裡的檔案路徑，例如 `/content/drive/MyDrive/錄音/0308.mp3`
* 資料夾路徑 → 辨識資料夾裡所有影音檔
* `.txt` 清單 → 一行一個路徑或網址
* YouTube 影片／播放清單網址，或其他 yt-dlp 支援的網址

繁體中文：建議勾選「轉成繁體」（用 OpenCC 轉換），**提示語留空**。提示語填指令式句子（例如「輸出為繁體中文」）容易讓辨識內容出錯或重複。


In [ ]:
#@title 1. 安裝套件 { display-mode: "form" }
!pip -q install -U faster-whisper pysubs2 yt-dlp opencc-python-reimplemented ollama sentencepiece sacremoses
# CTranslate2 需要 CUDA 12 的 cuBLAS / cuDNN 9，Colab 內建的版本不一定對得上
!pip -q install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import torch
if torch.cuda.is_available():
  print('✅ GPU:', torch.cuda.get_device_name(0))
else:
  print('⚠️ 沒有 GPU，會用 CPU 跑（很慢）。請到「執行階段 → 變更執行階段類型」選 T4 GPU 後重新執行。')


In [ ]:
#@title 2. 掛載雲端硬碟（選用） { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. 語音辨識 { display-mode: "form" }

#@markdown ### 來源
#@markdown 留空 = 從電腦上傳；也可填檔案路徑、資料夾、.txt 清單、YouTube 或其他網址
source = "" #@param {type:"string"}

#@markdown ### 辨識設定
language = "zh" #@param ["自動判斷", "zh", "en", "ja", "ko", "fr", "de", "es"] {allow-input: true}
model_size = "large-v3" #@param ["large-v3", "turbo", "large-v2", "medium", "small", "base"]
#@markdown 中文結果轉成台灣繁體（OpenCC s2twp）
traditional_chinese = True #@param {type:"boolean"}

#@markdown ### 輸出
output_format = "srt" #@param ["srt", "txt", "srt+txt"]
output_dir = "/content/output" #@param {type:"string"}
#@markdown 已經有結果檔的就跳過
skip_existing = False #@param {type:"boolean"}
#@markdown 完成後自動下載到電腦（多個檔案會打包成 zip）
auto_download = True #@param {type:"boolean"}
#@markdown 即時顯示辨識出來的每一句
show_segments = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 進階（通常不用改）
#@markdown 提示語：建議留空；可放專有名詞、人名，例如「寶可夢、皮卡丘」
initial_prompt = "" #@param {type:"string"}
vad_filter = True #@param {type:"boolean"}
vad_min_silence_ms = 500 #@param {type:"integer"}
beam_size = 5 #@param {type:"integer"}

import ctypes
import glob
import os
import site
import re
import subprocess
import urllib.request
import zipfile

import ctranslate2
import numpy as np
import pysubs2
from faster_whisper import WhisperModel
from google.colab import files
from tqdm.auto import tqdm
from yt_dlp import YoutubeDL

MEDIA_EXT = {'.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.opus', '.wma',
             '.mp4', '.mov', '.avi', '.mkv', '.webm', '.m4v', '.3gp'}
DOWNLOAD_DIR = '/content/_downloads'
UPLOAD_DIR = '/content/uploads'


def safe_name(name):
  """把標題轉成合法檔名"""
  name = re.sub(r'[\\/:*?"<>|\r\n\t]', '_', name).strip(' .')
  return name[:150] or 'untitled'


def is_url(s):
  return re.match(r'^https?://', s) is not None


def expand_sources(src):
  """把使用者輸入展開成 [(本機路徑或網址), ...]"""
  src = src.strip()
  if not src:
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    print('>> 請選擇要上傳的影音檔（可多選）')
    uploaded = files.upload()
    paths = []
    for name, data in uploaded.items():
      path = os.path.join(UPLOAD_DIR, name)
      with open(path, 'wb') as f:
        f.write(data)
      paths.append(path)
    return paths

  if is_url(src):
    return [src]

  if os.path.isdir(src):
    paths = sorted(p for p in glob.glob(os.path.join(src, '*'))
                   if os.path.splitext(p)[1].lower() in MEDIA_EXT)
    if not paths:
      print(f'⚠️ 資料夾裡沒有影音檔: {src}')
    return paths

  if src.lower().endswith('.txt'):
    if not os.path.exists(src):
      print(f'⚠️ 找不到清單檔: {src}')
      return []
    base = os.path.dirname(src)
    items = []
    with open(src, encoding='utf-8') as f:
      for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
          continue
        # 相對路徑以清單檔所在資料夾為準
        if not is_url(line) and not os.path.isabs(line) and not os.path.exists(line):
          line = os.path.join(base, line)
        items.extend(expand_sources(line))
    return items

  if os.path.exists(src):
    return [src]

  print(f'⚠️ 找不到檔案: {src}')
  return []


def download_url(url):
  """下載網址的音訊，回傳 [(音訊路徑, 標題), ...]（播放清單會有多個）"""
  os.makedirs(DOWNLOAD_DIR, exist_ok=True)

  m = re.search(r'(?:voca\.ro|vocaroo\.com)/(\w+)', url)
  if m:
    path = os.path.join(DOWNLOAD_DIR, f'vocaroo-{m.group(1)}.mp3')
    urllib.request.urlretrieve(f'https://media.vocaroo.com/mp3/{m.group(1)}', path)
    return [(path, f'vocaroo-{m.group(1)}')]

  opts = {
      'quiet': True,
      'no_warnings': True,
      'format': 'bestaudio/best',
      'outtmpl': os.path.join(DOWNLOAD_DIR, '%(id)s.%(ext)s'),
      'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3'}],
      'overwrites': True,
      'ignoreerrors': True,
  }
  with YoutubeDL(opts) as ydl:
    info = ydl.extract_info(url, download=True)
  if info is None:
    return []

  entries = info.get('entries') or [info]
  results = []
  for e in entries:
    if not e:
      continue
    downloads = e.get('requested_downloads') or []
    path = downloads[0].get('filepath') if downloads else None
    if path and os.path.exists(path):
      results.append((path, e.get('title') or e.get('id')))
    else:
      print(f"⚠️ 下載失敗: {e.get('title') or e.get('webpage_url')}")
  return results


def preload_cuda_libs():
  """先載入 pip 裝的 CUDA 12 函式庫，讓 CTranslate2 找得到 libcublas.so.12 / libcudnn*.so.9"""
  patterns = ['nvidia/cuda_runtime/lib/libcudart.so.12',
              'nvidia/cublas/lib/libcublas*.so.12',
              'nvidia/cudnn/lib/libcudnn*.so.9']
  libs = [lib for d in site.getsitepackages() for p in patterns
          for lib in sorted(glob.glob(os.path.join(d, p)))]
  # 函式庫之間有相依，載入失敗的下一輪再試，直到沒有進展
  while libs:
    failed = []
    for lib in libs:
      try:
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
      except OSError:
        failed.append(lib)
    if len(failed) == len(libs):
      break
    libs = failed


def get_model():
  """載入模型；同一個執行階段重複執行時沿用已載入的模型"""
  global _whisper_model, _whisper_model_key
  if ctranslate2.get_cuda_device_count() > 0:
    preload_cuda_libs()
    device, compute_type = 'cuda', 'float16'
  else:
    device, compute_type = 'cpu', 'int8'
  key = (model_size, device, compute_type)
  if globals().get('_whisper_model_key') != key:
    print(f'>> 載入模型 {model_size}（{device}, {compute_type}），第一次會先下載...')
    _whisper_model = WhisperModel(model_size, device=device, compute_type=compute_type)
    _whisper_model_key = key
  return _whisper_model


def output_paths(title):
  name = safe_name(title)
  exts = ['srt', 'txt'] if output_format == 'srt+txt' else [output_format]
  return [os.path.join(output_dir, f'{name}.{ext}') for ext in exts]


def load_audio(path):
  """用 ffmpeg 解碼成 16kHz 單聲道（不經過 PyAV，避免新版 av 跟 faster-whisper 不相容）"""
  cmd = ['ffmpeg', '-nostdin', '-loglevel', 'error', '-i', path,
         '-vn', '-f', 's16le', '-ac', '1', '-ar', '16000', '-']
  proc = subprocess.run(cmd, capture_output=True)
  if proc.returncode != 0:
    raise RuntimeError('ffmpeg 解碼失敗: ' + proc.stderr.decode('utf-8', 'ignore').strip())
  return np.frombuffer(proc.stdout, np.int16).astype(np.float32) / 32768.0


def transcribe(audio_path, title, converter):
  model = get_model()
  segments, info = model.transcribe(
      load_audio(audio_path),
      language=None if language == '自動判斷' else language,
      initial_prompt=initial_prompt.strip() or None,
      beam_size=beam_size,
      vad_filter=vad_filter,
      vad_parameters=dict(min_silence_duration_ms=vad_min_silence_ms),
  )
  print(f'   語言: {info.language}（{info.language_probability:.0%}）  長度: {info.duration/60:.1f} 分鐘')

  use_opencc = converter is not None and info.language == 'zh'
  results = []
  with tqdm(total=round(info.duration), unit='秒', disable=show_segments) as pbar:
    for seg in segments:
      text = seg.text.strip()
      if use_opencc:
        text = converter.convert(text)
      if not text:
        continue
      results.append({'start': seg.start, 'end': seg.end, 'text': text})
      if show_segments:
        print(f'   [{seg.start:7.2f} → {seg.end:7.2f}] {text}')
      pbar.update(max(0, round(seg.end) - pbar.n))

  saved = []
  for path in output_paths(title):
    if path.endswith('.srt'):
      pysubs2.load_from_whisper(results).save(path, encoding='utf-8')
    else:
      with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(r['text'] for r in results) + '\n')
    saved.append(path)
  return saved


# ── 主流程 ─────────────────────────────────────────────
os.makedirs(output_dir, exist_ok=True)

converter = None
if traditional_chinese:
  import opencc
  converter = opencc.OpenCC('s2twp')

sources = expand_sources(source)
if not sources:
  print('沒有要辨識的檔案。')

finished = []
for i, src in enumerate(sources, 1):
  print(f'\n[{i}/{len(sources)}] {src}')
  try:
    if is_url(src):
      print('>> 下載中...')
      items = download_url(src)
    else:
      items = [(src, os.path.splitext(os.path.basename(src))[0])]

    for audio_path, title in items:
      outs = output_paths(title)
      if skip_existing and all(os.path.exists(p) for p in outs):
        print(f'>> 已存在，跳過: {title}')
        finished.extend(outs)
        continue
      print(f'>> 辨識中: {title}')
      finished.extend(transcribe(audio_path, title, converter))
  except Exception as e:
    print(f'❌ 失敗: {type(e).__name__}: {e}')

print('\n' + '=' * 50)
if finished:
  print(f'✅ 完成 {len(finished)} 個檔案，存放在 {output_dir}')
  for p in finished:
    print('   ', p)
  if auto_download:
    if len(finished) == 1:
      files.download(finished[0])
    else:
      zip_path = '/content/字幕結果.zip'
      with zipfile.ZipFile(zip_path, 'w') as z:
        for p in finished:
          z.write(p, os.path.basename(p))
      files.download(zip_path)
  else:
    print('請到左側「檔案」欄位下載。')
else:
  print('❌ 沒有產生任何結果檔。')


In [ ]:
#@title 4. 字幕翻譯（中文 SRT → 英文） { display-mode: "form" }

#@markdown ### 來源
#@markdown 留空 = 從電腦上傳 .srt（可多選）；也可填 .srt 路徑或資料夾（例如 /content/output）
srt_source = "" #@param {type:"string"}

#@markdown ### 翻譯設定
#@markdown gemma4 / qwen2.5 用 Ollama 在 Colab 上跑，口語翻得好，第一次要下載模型（幾分鐘）；Helsinki 快但口語較弱
translate_model = "gemma4:e4b" #@param ["gemma4:e4b", "qwen2.5:7b", "Helsinki-NLP/opus-mt-zh-en"] {allow-input: true}
#@markdown 整份一次翻：整份字幕當純文字一次交給 LLM，前後文最完整（太長會自動改分批）
translate_mode = "整份一次翻" #@param ["整份一次翻", "分批翻"]
#@markdown 分批翻時，一次送幾句
batch_lines = 40 #@param {type:"integer"}

#@markdown ### 專有名詞
#@markdown 影片背景（讓模型知道在講什麼，例如：大阪京都自由行 vlog）
video_context = "日本旅遊 vlog" #@param {type:"string"}
#@markdown 日本地名、車站、寺廟、店名、料理用日文羅馬拼音（Asakusa），不用中文拼音（Qiancao）
japanese_romaji = True #@param {type:"boolean"}
#@markdown 名詞對照表：逗號分隔，例如 `淺草=Asakusa, 清水寺=Kiyomizu-dera, 一蘭=Ichiran`
glossary = "" #@param {type:"string"}
#@markdown 對照表很多的話，可以放在 .txt（一行一組 `中文=English`），填路徑
glossary_file = "" #@param {type:"string"}

#@markdown ### 輸出
subtitle_style = "英文" #@param ["英文", "中英雙語"]
translate_output_dir = "/content/output" #@param {type:"string"}
translate_auto_download = True #@param {type:"boolean"}

import gc
import glob
import os
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile

import pysubs2
from google.colab import files
from tqdm.auto import tqdm

BASE_PROMPT = ('You are a subtitle translator. '
               'Translate Chinese subtitles to natural, conversational English. '
               'Preserve the meaning across lines even if a sentence is split. '
               'Translate every numbered line and keep its number. '
               'Output only the numbered translations, one per line, '
               'in the format "N: translation". No extra text.')
ROMAJI_RULE = ('Japanese proper nouns (places, stations, temples, shrines, shops, brands, dishes) '
               'are written in Chinese characters in the source. Translate them to the standard '
               'Hepburn romanization used on Japanese signs and English maps, never Chinese pinyin. '
               'Examples: 淺草 → Asakusa, 清水寺 → Kiyomizu-dera, 心齋橋 → Shinsaibashi, '
               '道頓堀 → Dotonbori, 新宿站 → Shinjuku Station, 拉麵 → ramen.')
UPLOAD_DIR = '/content/uploads'
MAX_CTX = 32768  # Ollama 一次請求的最大 context（T4 上 7B~14B 模型撐得住）


def load_glossary():
  """讀取名詞對照，回傳 [(中文, English), ...]，長的詞排前面"""
  items = re.split(r'[,，;；\n]', glossary)
  if glossary_file.strip():
    if os.path.exists(glossary_file.strip()):
      with open(glossary_file.strip(), encoding='utf-8-sig') as f:
        items += f.read().splitlines()
    else:
      print(f'⚠️ 找不到對照表: {glossary_file}')
  pairs = {}
  for item in items:
    parts = re.split(r'\s*(?:=|＝|→)\s*', item.strip(), maxsplit=1)
    if len(parts) == 2 and parts[0] and parts[1]:
      pairs[parts[0]] = parts[1]
  return sorted(pairs.items(), key=lambda p: -len(p[0]))


def build_system_prompt(pairs):
  prompt = BASE_PROMPT
  if video_context.strip():
    prompt += f'\nBackground of the video: {video_context.strip()}'
  if japanese_romaji:
    prompt += '\n' + ROMAJI_RULE
  if pairs:
    prompt += '\nAlways translate these terms exactly as given:\n'
    prompt += '\n'.join(f'{zh} → {en}' for zh, en in pairs)
  return prompt


def check_glossary(zh_texts, en_texts, pairs):
  """對照表裡的詞有出現，但英文沒照著翻的，列出來提醒"""
  issues = [(n, zh, en) for n, (src, dst) in enumerate(zip(zh_texts, en_texts), 1)
            for zh, en in pairs if zh in src and en.lower() not in dst.lower()]
  if issues:
    print(f'⚠️ 有 {len(issues)} 處專有名詞沒照對照表翻，請檢查：')
    for n, zh, en in issues[:20]:
      print(f'   第 {n} 句: {zh} 應為 {en}')


def estimate_ctx(numbered):
  """粗估一次請求要多少 token：中文約一字一 token，英文輸出大約同量，再加 prompt"""
  return len(numbered) * 2 + 1024


def collect_srts(src):
  src = src.strip()
  if not src:
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    print('>> 請選擇要翻譯的 .srt（可多選）')
    paths = []
    for name, data in files.upload().items():
      path = os.path.join(UPLOAD_DIR, name)
      with open(path, 'wb') as f:
        f.write(data)
      paths.append(path)
    return paths
  if os.path.isdir(src):
    # 不要把已經翻好的 .en.srt / .zh-en.srt 再翻一次
    return sorted(p for p in glob.glob(os.path.join(src, '*.srt'))
                  if not re.search(r'\.(en|zh-en)\.srt$', p))
  if os.path.exists(src):
    return [src]
  print(f'⚠️ 找不到檔案: {src}')
  return []


def free_whisper():
  """辨識模型還佔著 GPU 的話先釋放，讓翻譯模型有足夠記憶體"""
  global _whisper_model, _whisper_model_key
  if globals().get('_whisper_model') is not None:
    _whisper_model = None
    _whisper_model_key = None
    gc.collect()


def ollama_alive():
  try:
    urllib.request.urlopen('http://127.0.0.1:11434', timeout=1)
    return True
  except Exception:
    return False


def start_ollama(model):
  if shutil.which('ollama') is None:
    print('>> 安裝 Ollama...')
    proc = subprocess.run('apt-get -qq install -y zstd && curl -fsSL https://ollama.com/install.sh | sh',
                          shell=True, capture_output=True, text=True)
    if proc.returncode != 0:
      raise RuntimeError('Ollama 安裝失敗:\n' + proc.stderr[-2000:])
  if not ollama_alive():
    print('>> 啟動 Ollama...')
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'),
                     stderr=subprocess.STDOUT)
    for _ in range(30):
      time.sleep(1)
      if ollama_alive():
        break
    else:
      raise RuntimeError('Ollama 啟動失敗，請看 /content/ollama.log')
  print(f'>> 準備模型 {model}（第一次要下載，請稍候）...')
  proc = subprocess.run(['ollama', 'pull', model], capture_output=True, text=True)
  if proc.returncode != 0:
    raise RuntimeError(f'下載模型 {model} 失敗:\n' + proc.stderr[-2000:])


def parse_numbered(raw):
  """解析 "N: 翻譯文字"，回傳 {索引: 翻譯}"""
  raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.S)
  out = {}
  duplicate = False
  for line in raw.splitlines():
    m = re.match(r'^\s*(\d+)\s*[:：.]\s*(.+)', line)
    if m:
      idx = int(m.group(1)) - 1
      if idx in out:
        duplicate = True
      out[idx] = m.group(2).strip()
  return {} if duplicate else out


def make_llm_translator(model):
  import ollama

  def ask(texts, idxs):
    numbered = '\n'.join(f'{i + 1}: {texts[i]}' for i in idxs)
    # Ollama 預設 context 很短，整份送會被截掉，依內容長度放大（以 4096 為單位）
    num_ctx = min(MAX_CTX, max(8192, -(-estimate_ctx(numbered) // 4096) * 4096))
    resp = ollama.chat(model=model, options={'temperature': 0.2, 'num_ctx': num_ctx}, messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': numbered},
    ])
    wanted = set(idxs)
    parsed = parse_numbered(resp['message']['content'])
    return parsed if set(parsed) <= wanted else {}

  def translate(texts):
    total = len(texts)
    size = batch_lines
    if translate_mode == '整份一次翻':
      est = estimate_ctx('\n'.join(f'{i + 1}: {t}' for i, t in enumerate(texts)))
      if est <= MAX_CTX:
        size = total
        print(f'>> 整份 {total} 句一次送出翻譯（較長的字幕要等幾分鐘）...')
      else:
        size = max(batch_lines, total * MAX_CTX * 9 // (est * 10))  # 留 10% 餘裕
        print(f'>> 字幕太長，一次放不下，改成每批 {size} 句')
    result = {}
    for s in tqdm(range(0, total, size), desc='翻譯中', unit='批'):
      result.update(ask(texts, list(range(s, min(s + size, total)))))
    # LLM 偶爾會漏句，漏掉的用小批次再翻一次
    missing = [i for i in range(total) if i not in result]
    if missing:
      print(f'>> 補翻漏掉的 {len(missing)} 句...')
      for s in range(0, len(missing), 10):
        result.update(ask(texts, missing[s:s + 10]))
    still = [i + 1 for i in range(total) if i not in result]
    if still:
      print(f'⚠️ 第 {still} 句沒翻成功，保留原文，請手動檢查')
    return [result.get(i, texts[i] + ' [TRANSLATION_FAILED]') for i in range(total)]

  return translate


def make_helsinki_translator(model_id):
  import torch
  from transformers import MarianMTModel, MarianTokenizer
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print(f'>> 載入 {model_id}...')
  tokenizer = MarianTokenizer.from_pretrained(model_id)
  model = MarianMTModel.from_pretrained(model_id).to(device)

  def translate(texts):
    # Helsinki 不吃提示語，對照表的詞先直接換成英文
    for zh, en in GLOSSARY:
      texts = [t.replace(zh, f' {en} ') for t in texts]
    out = []
    for s in tqdm(range(0, len(texts), 16), desc='翻譯中', unit='批'):
      inputs = tokenizer(texts[s:s + 16], return_tensors='pt', padding=True,
                         truncation=True, max_length=512).to(device)
      with torch.no_grad():
        generated = model.generate(**inputs)
      out += tokenizer.batch_decode(generated, skip_special_tokens=True)
    return out

  return translate


# ── 主流程 ─────────────────────────────────────────────
os.makedirs(translate_output_dir, exist_ok=True)
srts = collect_srts(srt_source)

GLOSSARY = load_glossary()
SYSTEM_PROMPT = build_system_prompt(GLOSSARY)
if GLOSSARY:
  print(f'>> 名詞對照 {len(GLOSSARY)} 組: ' + '、'.join(f'{zh}={en}' for zh, en in GLOSSARY[:10]))

translate = None
if srts:
  free_whisper()
  if translate_model.startswith('Helsinki-NLP/'):
    translate = make_helsinki_translator(translate_model)
  else:
    start_ollama(translate_model)
    translate = make_llm_translator(translate_model)
else:
  print('沒有要翻譯的字幕檔。')

translated_files = []
for i, path in enumerate(srts, 1):
  print(f'\n[{i}/{len(srts)}] {path}')
  try:
    subs = pysubs2.load(path, encoding='utf-8-sig')
    texts = [ev.plaintext.strip().replace('\n', ' ') for ev in subs]
    keep = [j for j, t in enumerate(texts) if t]
    english = translate([texts[j] for j in keep])
    check_glossary([texts[j] for j in keep], english, GLOSSARY)
    for j, en in zip(keep, english):
      ev = subs[j]
      if en.endswith('[TRANSLATION_FAILED]'):
        ev.plaintext = ev.plaintext + ' [TRANSLATION_FAILED]'
      else:
        ev.plaintext = f'{ev.plaintext.strip()}\n{en}' if subtitle_style == '中英雙語' else en

    for j in keep[:5]:
      print(f'   {texts[j]}  →  {subs[j].plaintext.splitlines()[-1]}')

    suffix = 'zh-en' if subtitle_style == '中英雙語' else 'en'
    base = os.path.splitext(os.path.basename(path))[0]
    out = os.path.join(translate_output_dir, f'{base}.{suffix}.srt')
    subs.save(out, encoding='utf-8')
    translated_files.append(out)
  except Exception as e:
    print(f'❌ 失敗: {type(e).__name__}: {e}')

print('\n' + '=' * 50)
if translated_files:
  print(f'✅ 完成 {len(translated_files)} 個檔案，存放在 {translate_output_dir}')
  for p in translated_files:
    print('   ', p)
  if translate_auto_download:
    if len(translated_files) == 1:
      files.download(translated_files[0])
    else:
      zip_path = '/content/翻譯結果.zip'
      with zipfile.ZipFile(zip_path, 'w') as z:
        for p in translated_files:
          z.write(p, os.path.basename(p))
      files.download(zip_path)
  else:
    print('請到左側「檔案」欄位下載。')
elif srts:
  print('❌ 沒有產生任何翻譯檔。')
